## Creates the **Epidemiological Timeline** for a specific region by using the available data on WNV Cases ##

In [1]:
import pandas as pd
import numpy as np

enc = 'utf-8'
dec = 'greek8'

pd.options.display.max_columns = None
pd.options.display.max_rows = 10

In [2]:
NUTS0 = 'GR'
NUTS2 = 'Peloponnese'
NUTS2_EL = 'πελοποννησου'

In [3]:
data = pd.read_csv(f'../../data/{NUTS0}_WNV_cases_2010-2023_processed.csv', encoding = enc)
data.head(5)

,onset of symptoms,year,month,day,nuts2,nuts3,cases
0,2010-07-06,2010,7,6,κεντρικης μακεδονιας,σερρων,1
1,2010-07-16,2010,7,16,κεντρικης μακεδονιας,κιλκις,1
2,2010-07-18,2010,7,18,κεντρικης μακεδονιας,πελλας,1
3,2010-07-19,2010,7,19,κεντρικης μακεδονιας,θεσσαλονικης,2
4,2010-07-20,2010,7,20,κεντρικης μακεδονιας,ημαθιας,1


In [4]:
data.shape

(1431, 7)

In [5]:
df = data[data['nuts2'].isin([NUTS2_EL])].copy()

In [6]:
df.reset_index(drop = True, inplace = True)

In [7]:
df.drop(columns = ['onset of symptoms', 'day'], inplace = True)

In [8]:
df

,year,month,nuts2,nuts3,cases
0,2017,6,πελοποννησου,αργολιδας,1
1,2017,6,πελοποννησου,αργολιδας,1
2,2017,7,πελοποννησου,αργολιδας,1
3,2017,7,πελοποννησου,αργολιδας,1
4,2017,7,πελοποννησου,αργολιδας,1
...,...,...,...,...,...
28,2017,8,πελοποννησου,κορινθιας,1
29,2017,9,πελοποννησου,κορινθιας,1
30,2018,7,πελοποννησου,κορινθιας,1
31,2018,8,πελοποννησου,αργολιδας,1


In [9]:
df.cases.sum()

45

In [10]:
df_grouped = df.groupby(['year', 'month', 'nuts2', 'nuts3'], as_index = False)
df = df_grouped.sum()

In [11]:
df

,year,month,nuts2,nuts3,cases
0,2017,6,πελοποννησου,αργολιδας,2
1,2017,7,πελοποννησου,αργολιδας,30
2,2017,7,πελοποννησου,αρκαδιας,1
3,2017,8,πελοποννησου,αργολιδας,6
4,2017,8,πελοποννησου,κορινθιας,2
5,2017,9,πελοποννησου,κορινθιας,1
6,2018,7,πελοποννησου,κορινθιας,1
7,2018,8,πελοποννησου,αργολιδας,1
8,2023,10,πελοποννησου,αργολιδας,1


In [12]:
df.cases.sum()

45

In [13]:
print(np.sort(df.cases.unique()))

[ 1  2  6 30]


In [14]:
wnv_lau1_list = df['nuts3'].drop_duplicates().sort_values().tolist()

with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2023_NUTS3_Units_Processed.txt', 'w', encoding=enc) as f:
    for item in wnv_lau1_list:
        f.write("%s\n" % item)
  
print(f"Number of NUTS3 Units in {NUTS2} (from cases): {len(wnv_lau1_list)}")

Number of NUTS3 Units in Peloponnese (from cases): 3


In [15]:
df['date_string'] = df['month'].astype(str) + '-' + df['year'].astype(str)

# Convert 'date_string' column to datetime format
df['dt_placement'] = pd.to_datetime(df['date_string'], format='%m-%Y').dt.to_period('M')
df.drop(columns=['date_string'], inplace = True)

In [16]:
df

,year,month,nuts2,nuts3,cases,dt_placement
0,2017,6,πελοποννησου,αργολιδας,2,2017-06
1,2017,7,πελοποννησου,αργολιδας,30,2017-07
2,2017,7,πελοποννησου,αρκαδιας,1,2017-07
3,2017,8,πελοποννησου,αργολιδας,6,2017-08
4,2017,8,πελοποννησου,κορινθιας,2,2017-08
5,2017,9,πελοποννησου,κορινθιας,1,2017-09
6,2018,7,πελοποννησου,κορινθιας,1,2018-07
7,2018,8,πελοποννησου,αργολιδας,1,2018-08
8,2023,10,πελοποννησου,αργολιδας,1,2023-10


In [17]:
df.cases.sum()

45

In [18]:
case_months = df['month'].drop_duplicates().sort_values().tolist()
case_months

[6, 7, 8, 9, 10]

In [19]:
case_years = df['year'].drop_duplicates().sort_values().tolist()
case_years

[2017, 2018, 2023]

In [20]:
with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2023_NUTS3_Cases_Dates.txt', 'w', encoding=enc) as f:

    f.write("%s: " % 'months')
    for item in case_months:
        f.write("%s, " % item)

    f.write("\n")

    f.write("%s: " % 'years')
    for item in case_years:
        f.write("%s, " % item)

In [21]:
df.drop(columns=['month', 'year'], inplace = True)

In [22]:
## Creating DataFrame to hold negative examples

df_timeline = pd.DataFrame(columns = ['nuts2', 'nuts3', 'dt_placement', 'cases'])
timeline = []

for nuts3 in df['nuts3'].drop_duplicates().sort_values():
    for year in case_years:
        for month in case_months:
            timeline.append({'nuts2' : NUTS2_EL, 'nuts3' : nuts3, 'dt_placement' : f"{year:04}-{month:02}", 'cases' : 0})
            
df_timeline = pd.DataFrame(timeline)

In [23]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,πελοποννησου,αργολιδας,2017-06,0
1,πελοποννησου,αργολιδας,2017-07,0
2,πελοποννησου,αργολιδας,2017-08,0
3,πελοποννησου,αργολιδας,2017-09,0
4,πελοποννησου,αργολιδας,2017-10,0
...,...,...,...,...
40,πελοποννησου,κορινθιας,2023-06,0
41,πελοποννησου,κορινθιας,2023-07,0
42,πελοποννησου,κορινθιας,2023-08,0
43,πελοποννησου,κορινθιας,2023-09,0


In [24]:
df_timeline['dt_placement']= pd.to_datetime(df_timeline['dt_placement'], format='%Y-%m').dt.to_period('M')

In [25]:
df.reset_index(inplace = True, drop=True)
df_timeline.reset_index(inplace = True, drop=True)

In [26]:
rearranged_cols = ['nuts2',	'nuts3', 'dt_placement', 'cases']

# Reindex the DataFrame with the desired column order
df = df.reindex(columns=rearranged_cols)

In [27]:
df

,nuts2,nuts3,dt_placement,cases
0,πελοποννησου,αργολιδας,2017-06,2
1,πελοποννησου,αργολιδας,2017-07,30
2,πελοποννησου,αρκαδιας,2017-07,1
3,πελοποννησου,αργολιδας,2017-08,6
4,πελοποννησου,κορινθιας,2017-08,2
5,πελοποννησου,κορινθιας,2017-09,1
6,πελοποννησου,κορινθιας,2018-07,1
7,πελοποννησου,αργολιδας,2018-08,1
8,πελοποννησου,αργολιδας,2023-10,1


In [28]:
df_timeline

,nuts2,nuts3,dt_placement,cases
0,πελοποννησου,αργολιδας,2017-06,0
1,πελοποννησου,αργολιδας,2017-07,0
2,πελοποννησου,αργολιδας,2017-08,0
3,πελοποννησου,αργολιδας,2017-09,0
4,πελοποννησου,αργολιδας,2017-10,0
...,...,...,...,...
40,πελοποννησου,κορινθιας,2023-06,0
41,πελοποννησου,κορινθιας,2023-07,0
42,πελοποννησου,κορινθιας,2023-08,0
43,πελοποννησου,κορινθιας,2023-09,0


In [29]:
merged_timeline = pd.merge(df_timeline, df, how ='left', on=['nuts2','nuts3','dt_placement'])

In [30]:
merged_timeline

,nuts2,nuts3,dt_placement,cases_x,cases_y
0,πελοποννησου,αργολιδας,2017-06,0,2.0
1,πελοποννησου,αργολιδας,2017-07,0,30.0
2,πελοποννησου,αργολιδας,2017-08,0,6.0
3,πελοποννησου,αργολιδας,2017-09,0,NaN
4,πελοποννησου,αργολιδας,2017-10,0,NaN
...,...,...,...,...,...
40,πελοποννησου,κορινθιας,2023-06,0,NaN
41,πελοποννησου,κορινθιας,2023-07,0,NaN
42,πελοποννησου,κορινθιας,2023-08,0,NaN
43,πελοποννησου,κορινθιας,2023-09,0,NaN


In [31]:
merged_timeline['cases_y'] = merged_timeline['cases_y'].fillna(0)

In [32]:
merged_timeline['cases'] = merged_timeline['cases_x'] + merged_timeline['cases_y']
merged_timeline = merged_timeline.drop(columns=['cases_x', 'cases_y'])
merged_timeline['cases'] = merged_timeline['cases'].astype(int)

In [33]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,πελοποννησου,αργολιδας,2017-06,2
1,πελοποννησου,αργολιδας,2017-07,30
2,πελοποννησου,αργολιδας,2017-08,6
3,πελοποννησου,αργολιδας,2017-09,0
4,πελοποννησου,αργολιδας,2017-10,0
...,...,...,...,...
40,πελοποννησου,κορινθιας,2023-06,0
41,πελοποννησου,κορινθιας,2023-07,0
42,πελοποννησου,κορινθιας,2023-08,0
43,πελοποννησου,κορινθιας,2023-09,0


In [34]:
merged_timeline['cases'].value_counts().sort_index()

cases
0     36
1      5
2      2
6      1
30     1
Name: count, dtype: int64

In [35]:
merged_timeline

,nuts2,nuts3,dt_placement,cases
0,πελοποννησου,αργολιδας,2017-06,2
1,πελοποννησου,αργολιδας,2017-07,30
2,πελοποννησου,αργολιδας,2017-08,6
3,πελοποννησου,αργολιδας,2017-09,0
4,πελοποννησου,αργολιδας,2017-10,0
...,...,...,...,...
40,πελοποννησου,κορινθιας,2023-06,0
41,πελοποννησου,κορινθιας,2023-07,0
42,πελοποννησου,κορινθιας,2023-08,0
43,πελοποννησου,κορινθιας,2023-09,0


In [36]:
merged_timeline.cases.sum()

45

In [37]:
merged_timeline.nuts3.unique()

array(['αργολιδας', 'αρκαδιας', 'κορινθιας'], dtype=object)

In [38]:
merged_timeline.nuts2.unique()

array(['πελοποννησου'], dtype=object)

In [39]:
merged_timeline.drop(columns=['nuts2'], inplace = True)
merged_timeline.rename(columns={'nuts3': 'NUTS3_NAME'}, inplace= True)

In [40]:
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('αργολιδας','αργολιδα, αρκαδια'))
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('αρκαδιας','αργολιδα, αρκαδια'))
merged_timeline['NUTS3_NAME'] = merged_timeline['NUTS3_NAME'].apply(lambda x : x.replace('κορινθιας','κορινθια'))

In [41]:
merged_timeline['NUTS2_NAME'] = 'πελοποννησος'

In [42]:
merged_timeline = merged_timeline[['NUTS2_NAME', 'NUTS3_NAME', 'dt_placement', 'cases']]
merged_timeline

,NUTS2_NAME,NUTS3_NAME,dt_placement,cases
0,πελοποννησος,"αργολιδα, αρκαδια",2017-06,2
1,πελοποννησος,"αργολιδα, αρκαδια",2017-07,30
2,πελοποννησος,"αργολιδα, αρκαδια",2017-08,6
3,πελοποννησος,"αργολιδα, αρκαδια",2017-09,0
4,πελοποννησος,"αργολιδα, αρκαδια",2017-10,0
...,...,...,...,...
40,πελοποννησος,κορινθια,2023-06,0
41,πελοποννησος,κορινθια,2023-07,0
42,πελοποννησος,κορινθια,2023-08,0
43,πελοποννησος,κορινθια,2023-09,0


In [43]:
merged_timeline.cases.sum()

45

In [44]:
grouped = merged_timeline.groupby(['NUTS2_NAME', 'NUTS3_NAME',	'dt_placement'], as_index = False)
timeline_grouped = grouped.sum()
timeline_grouped

,NUTS2_NAME,NUTS3_NAME,dt_placement,cases
0,πελοποννησος,"αργολιδα, αρκαδια",2017-06,2
1,πελοποννησος,"αργολιδα, αρκαδια",2017-07,31
2,πελοποννησος,"αργολιδα, αρκαδια",2017-08,6
3,πελοποννησος,"αργολιδα, αρκαδια",2017-09,0
4,πελοποννησος,"αργολιδα, αρκαδια",2017-10,0
...,...,...,...,...
25,πελοποννησος,κορινθια,2023-06,0
26,πελοποννησος,κορινθια,2023-07,0
27,πελοποννησος,κορινθια,2023-08,0
28,πελοποννησος,κορινθια,2023-09,0


In [45]:
timeline_grouped.cases.sum()

45

In [46]:
merged_timeline.to_csv(f"../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Cases_NUTS3_2010-2023.csv", encoding = enc, index = False)